# RAGBench-KR error-analysis export smoke test

This notebook uses a deterministic **public synthetic fixture only**. It does not access a database, gold questions, private documents, or a provider. Its values are test data and must not be cited as benchmark findings. The committed notebook has no execution outputs.

In [ ]:
import hashlib
from decimal import Decimal
from pathlib import Path
from tempfile import TemporaryDirectory

from ragbench.analysis import AnalysisBundle, ExportRequest, export_analysis

In [ ]:
def sha(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


configs = []
results = []
usage = []
failures = []
for index, mode in enumerate(("standard", "enhanced")):
    config_hash = sha(f"synthetic-config-{index}")
    experiment_id = f"{config_hash}-20260814T00000{index}.000000Z"
    configs.append(
        {
            "experiment_id": experiment_id,
            "config_hash": config_hash,
            "parse_mode": mode,
            "chunk_strategy": "fixed-512-64",
            "retriever": "dense",
            "top_k": 5,
            "prompt_version": "v1",
            "model_id": "synthetic-model-v1",
        }
    )
    for question_index in range(30):
        question_id = f"public-synthetic-{question_index:03d}"
        question_type = "table" if question_index % 2 else "fact"
        score = Decimal("0.50") + Decimal(index) * Decimal("0.10")
        results.append(
            {
                "experiment_id": experiment_id,
                "question_id": question_id,
                "question_type": question_type,
                "correctness": score,
                "faithfulness": score,
                "citation": score,
                "abstention": Decimal("0.75"),
                "hit": Decimal("1"),
                "recall": Decimal("0.80"),
                "mrr": Decimal("0.75"),
                "latency_ms": 400 + index * 50 + question_index,
            }
        )
        usage.append(
            {
                "experiment_id": experiment_id,
                "question_id": question_id,
                "question_type": question_type,
                "operation": "generate",
                "model_id": "synthetic-model-v1",
                "estimated_cost_usd": Decimal("0.001"),
                "cached": question_index % 3 == 0,
            }
        )
        failures.append(
            {
                "experiment_id": experiment_id,
                "question_id": question_id,
                "question_type": question_type,
                "primary": "TABLE_ERROR" if question_type == "table" else "RETRIEVAL_MISS",
                "secondary": None,
            }
        )

In [ ]:
bundle = AnalysisBundle.model_validate(
    {
        "schema_version": "analysis-input-v1",
        "cohort_hash": sha("public-synthetic-cohort"),
        "data_version": "public-synthetic-v1",
        "code_version": "fc6e5aa",
        "configs": configs,
        "results": results,
        "usage": usage,
        "failures": failures,
    }
)
request = ExportRequest(bundles=(bundle,), public_salt="not-a-release-salt", failure_sample_size=50)

In [ ]:
with TemporaryDirectory() as directory:
    destination = Path(directory).resolve() / "analysis"
    manifest = export_analysis(request, destination)
    assert manifest.claim_status == "PENDING_EVIDENCE"
    assert len(manifest.tables) == 13
    assert (destination / "tables" / "leaderboard.parquet").read_bytes()[:4] == b"PAR1"
    assert all((destination / path).is_file() for path in manifest.files)